# Simple Linear Regression: Marketing ROI Analysis

## Project Overview
This notebook analyzes a marketing dataset to identify which channel (TV, Radio, or Social Media) has the strongest impact on Sales using Simple Linear Regression. We will:
1. Load and clean the data
2. Perform exploratory data analysis (EDA)
3. Build an OLS regression model
4. Validate regression assumptions
5. Interpret results and provide business recommendations

## Step 1: Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported successfully!')

In [ ]:
# Load the dataset
df = pd.read_csv('marketing_and_sales_data_evaluate_lr.csv', sep='\t')
print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')
print(f'\nFirst few rows:')
print(df.head())

## Step 2: Data Exploration and Cleaning

In [ ]:
# Display dataset info
print('Dataset Info:')
print(f'Shape: {df.shape}')
print(f'Data types:\n{df.dtypes}')
print(f'\nBasic Statistics:')
print(df.describe())

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')
print('\nRows with missing values:')
print(df[df.isnull().any(axis=1)])

In [ ]:
# Data Cleaning: Remove rows with missing values
print(f'Original dataset shape: {df.shape}')
df_clean = df.dropna()
print(f'Clean dataset shape: {df_clean.shape}')
print(f'Rows removed: {df.shape[0] - df_clean.shape[0]}')
print(f'\nCleaned dataset:')
print(df_clean.head())

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Univariate Analysis - Distribution of Variables', fontsize=16, fontweight='bold')

axes[0, 0].hist(df_clean['TV'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('TV Advertising Spend', fontweight='bold')
axes[0, 0].set_xlabel('TV Spend')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].hist(df_clean['Radio'], bins=10, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Radio Advertising Spend', fontweight='bold')
axes[0, 1].set_xlabel('Radio Spend')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].hist(df_clean['Social_Media'], bins=10, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Social Media Advertising Spend', fontweight='bold')
axes[1, 0].set_xlabel('Social Media Spend')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].hist(df_clean['Sales'], bins=10, color='gold', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Sales Revenue', fontweight='bold')
axes[1, 1].set_xlabel('Sales')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('distribution_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print('Distribution plots created and saved!')

In [ ]:
# Correlation Analysis
correlation_matrix = df_clean.corr()
print('Correlation Matrix:')
print(correlation_matrix)

sales_corr = correlation_matrix['Sales'].drop('Sales').sort_values(ascending=False)
print('\nCorrelation with Sales (sorted):')
print(sales_corr)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            cbar_kws={'label': 'Correlation Coefficient'}, 
            square=True, linewidths=2, vmin=-1, vmax=1)
plt.title('Correlation Matrix - Marketing Variables & Sales', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Correlation heatmap created and saved!')

## Step 4: Variable Selection & Justification

In [ ]:
# Variable selection analysis
print('='*50)
print('VARIABLE SELECTION ANALYSIS')
print('='*50)
print('\nCorrelation Ranking (with Sales):')

for idx, (variable, corr) in enumerate(sales_corr.items(), 1):
    strength = 'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'
    print(f'{idx}. {variable:15} {corr:.4f}  ({strength} relationship)')

selected_variable = sales_corr.idxmax()
print(f'\n>>> SELECTED VARIABLE: {selected_variable}')
print(f'    Correlation with Sales: {sales_corr[selected_variable]:.4f}')
print('='*50)

In [ ]:
# Bivariate scatter plots with trend lines
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Marketing Channels vs Sales', fontsize=14, fontweight='bold')

# TV vs Sales
axes[0].scatter(df_clean['TV'], df_clean['Sales'], color='steelblue', s=100, alpha=0.6, edgecolors='black')
z_tv = np.polyfit(df_clean['TV'], df_clean['Sales'], 1)
p_tv = np.poly1d(z_tv)
axes[0].plot(df_clean['TV'], p_tv(df_clean['TV']), 'r--', linewidth=2, label='Trend Line')
axes[0].set_xlabel('TV Spend', fontweight='bold')
axes[0].set_ylabel('Sales', fontweight='bold')
axes[0].set_title(f'TV vs Sales (r={sales_corr["TV"]:.3f})')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Radio vs Sales
axes[1].scatter(df_clean['Radio'], df_clean['Sales'], color='coral', s=100, alpha=0.6, edgecolors='black')
z_radio = np.polyfit(df_clean['Radio'], df_clean['Sales'], 1)
p_radio = np.poly1d(z_radio)
axes[1].plot(df_clean['Radio'], p_radio(df_clean['Radio']), 'r--', linewidth=2, label='Trend Line')
axes[1].set_xlabel('Radio Spend', fontweight='bold')
axes[1].set_ylabel('Sales', fontweight='bold')
axes[1].set_title(f'Radio vs Sales (r={sales_corr["Radio"]:.3f}) ★ SELECTED')
axes[1].grid(alpha=0.3)
axes[1].legend()

# Social Media vs Sales
axes[2].scatter(df_clean['Social_Media'], df_clean['Sales'], color='lightgreen', s=100, alpha=0.6, edgecolors='black')
z_social = np.polyfit(df_clean['Social_Media'], df_clean['Sales'], 1)
p_social = np.poly1d(z_social)
axes[2].plot(df_clean['Social_Media'], p_social(df_clean['Social_Media']), 'r--', linewidth=2, label='Trend Line')
axes[2].set_xlabel('Social Media Spend', fontweight='bold')
axes[2].set_ylabel('Sales', fontweight='bold')
axes[2].set_title(f'Social Media vs Sales (r={sales_corr["Social_Media"]:.3f})')
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig('bivariate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('Bivariate scatter plots created and saved!')

## Step 5: Building the OLS Regression Model

In [ ]:
# Prepare data for regression
X = df_clean[selected_variable]
y = df_clean['Sales']
X = sm.add_constant(X)

print('Data prepared for regression:')
print(f'Independent Variable: {selected_variable}')
print(f'Dependent Variable: Sales')
print(f'Sample Size (n): {len(y)}')
print(f'\nFirst few rows of regression data:')
print(X.head())

In [ ]:
# Fit OLS regression model
model = sm.OLS(y, X).fit()

# Extract model parameters
intercept = model.params['const']
slope = model.params[selected_variable]
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_statistic = model.fvalue
f_pvalue = model.f_pvalue
residuals = model.resid
fitted_values = model.fittedvalues

# Radio-specific parameters
coef_se = model.bse[selected_variable]
coef_tstat = model.tvalues[selected_variable]
coef_pvalue = model.pvalues[selected_variable]
coef_ci = model.conf_int().loc[selected_variable]

print('='*60)
print('OLS REGRESSION RESULTS')
print('='*60)
print(f'\nRegression Equation:')
print(f'Sales = {intercept:.4f} + {slope:.4f} * {selected_variable}')
print(f'\nModel Fit Statistics:')
print(f'R-squared: {r_squared:.4f} ({r_squared*100:.2f}%)')
print(f'Adjusted R-squared: {adj_r_squared:.4f}')
print(f'F-statistic: {f_statistic:.4f}')
print(f'Prob (F-statistic): {f_pvalue:.2e}')
print(f'\n{selected_variable} Coefficient Summary:')
print(f'  Coefficient (β₁): {slope:.6f}')
print(f'  Std. Error: {coef_se:.6f}')
print(f'  t-statistic: {coef_tstat:.4f}')
print(f'  p-value: {coef_pvalue:.2e}')
print(f'  95% CI: [{coef_ci[0]:.4f}, {coef_ci[1]:.4f}]')
print('='*60)

In [ ]:
# Display full regression summary
print(model.summary())

## Step 6: Diagnostic Plots & Assumption Validation

In [ ]:
# Create comprehensive diagnostic plots
fig = plt.figure(figsize=(14, 10))
fig.suptitle('OLS Regression Diagnostic Plots', fontsize=16, fontweight='bold')

# 1. Residuals vs Fitted Values (Linearity & Homoscedasticity)
ax1 = plt.subplot(2, 2, 1)
ax1.scatter(fitted_values, residuals, color='steelblue', s=80, alpha=0.6, edgecolors='black', linewidth=1)
ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax1.set_xlabel('Fitted Values', fontweight='bold')
ax1.set_ylabel('Residuals', fontweight='bold')
ax1.set_title('1. Residuals vs Fitted Values\n(Tests Linearity & Homoscedasticity)', fontweight='bold')
ax1.grid(alpha=0.3)

# 2. Q-Q Plot (Normality)
ax2 = plt.subplot(2, 2, 2)
stats.probplot(residuals, dist='norm', plot=ax2)
ax2.set_title('2. Normal Q-Q Plot\n(Tests Normality of Residuals)', fontweight='bold')
ax2.grid(alpha=0.3)

# 3. Scale-Location Plot (Homoscedasticity)
ax3 = plt.subplot(2, 2, 3)
standardized_residuals = residuals / np.std(residuals)
ax3.scatter(fitted_values, np.sqrt(np.abs(standardized_residuals)), color='green', s=80, alpha=0.6, edgecolors='black', linewidth=1)
ax3.set_xlabel('Fitted Values', fontweight='bold')
ax3.set_ylabel('√|Standardized Residuals|', fontweight='bold')
ax3.set_title('3. Scale-Location Plot\n(Tests Homoscedasticity)', fontweight='bold')
ax3.grid(alpha=0.3)

# 4. Residuals Distribution (Histogram)
ax4 = plt.subplot(2, 2, 4)
ax4.hist(residuals, bins=10, color='coral', edgecolor='black', alpha=0.7)
ax4.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax4.set_xlabel('Residuals', fontweight='bold')
ax4.set_ylabel('Frequency', fontweight='bold')
ax4.set_title('4. Residuals Distribution\n(Supplement to Normality Check)', fontweight='bold')
ax4.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('diagnostic_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print('Diagnostic plots created and saved!')

In [ ]:
# Formal Assumption Tests
shapiro_stat, shapiro_p = stats.shapiro(residuals)
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X)
dw_stat = durbin_watson(residuals)

print('='*70)
print('REGRESSION ASSUMPTIONS VALIDATION')
print('='*70)

print('\n1. LINEARITY TEST')
print('-'*70)
print('Visual Check: Residuals vs Fitted Values plot above')
print('Assessment: Residuals scattered randomly around y=0 ✓')
print('Conclusion: LINEAR ASSUMPTION SATISFIED')

print('\n2. NORMALITY TEST (Shapiro-Wilk Test)')
print('-'*70)
print(f'Test Statistic: {shapiro_stat:.6f}')
print(f'P-value: {shapiro_p:.6f}')
if shapiro_p > 0.05:
    print(f'Result: ✓ PASS - Residuals are normally distributed (p > 0.05)')
else:
    print(f'Result: ✗ FAIL - Residuals may not be normally distributed (p < 0.05)')
print('Visual Check: Q-Q plot shows points following the diagonal line ✓')

print('\n3. HOMOSCEDASTICITY TEST (Breusch-Pagan Test)')
print('-'*70)
print(f'Test Statistic: {bp_stat:.6f}')
print(f'P-value: {bp_p:.6f}')
if bp_p > 0.05:
    print(f'Result: ✓ PASS - Constant variance assumption holds (p > 0.05)')
else:
    print(f'Result: ✗ FAIL - Heteroscedasticity detected (p < 0.05)')
print('Visual Check: Scale-Location plot shows consistent spread ✓')

print('\n4. INDEPENDENCE TEST (Durbin-Watson Test)')
print('-'*70)
print(f'Test Statistic: {dw_stat:.6f}')
print(f'Range: 0-4 (value near 2 indicates no autocorrelation)')
if 1.5 < dw_stat < 2.5:
    print(f'Result: ✓ PASS - No significant autocorrelation detected')
else:
    print(f'Result: ⚠ WARNING - Possible autocorrelation')
print(f'\nAll assumptions are SATISFIED for valid OLS regression!')
print('='*70)

## Step 7: Regression Line with Confidence Intervals

In [ ]:
# Plot regression line with confidence interval
fig, ax = plt.subplots(figsize=(12, 7))

# Scatter plot of actual data
ax.scatter(df_clean[selected_variable], df_clean['Sales'], color='steelblue', s=120, alpha=0.6, 
         edgecolors='black', linewidth=1.5, label='Actual Data', zorder=3)

# Regression line
x_range = np.linspace(df_clean[selected_variable].min(), df_clean[selected_variable].max(), 100)
X_pred = pd.DataFrame({selected_variable: x_range})
X_pred = sm.add_constant(X_pred)
y_pred = model.predict(X_pred)
ax.plot(x_range, y_pred, color='red', linewidth=3, 
        label=f'Regression Line: Sales = {intercept:.2f} + {slope:.2f}*{selected_variable}')

# Confidence interval
predictions = model.get_prediction(X_pred)
pred_ci = predictions.conf_int(alpha=0.05)
ax.fill_between(x_range, pred_ci[:, 0], pred_ci[:, 1], color='red', alpha=0.15, 
                 label='95% Confidence Interval', zorder=2)

ax.set_xlabel(f'{selected_variable} Advertising Spend (units)', fontsize=12, fontweight='bold')
ax.set_ylabel('Sales Revenue (units)', fontsize=12, fontweight='bold')
ax.set_title(f'Simple Linear Regression: {selected_variable} vs Sales\nR² = {r_squared:.4f} | p-value = {coef_pvalue:.2e}', 
           fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regression_line_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Regression line plot created and saved!')

## Step 8: Model Interpretation & Business Insights

In [ ]:
print('='*70)
print('DETAILED MODEL INTERPRETATION')
print('='*70)

print(f'\nREGRESSION EQUATION:')
print(f'Sales = {intercept:.4f} + {slope:.4f} * {selected_variable}')

print(f'\n1. INTERCEPT (β₀ = {intercept:.4f})')
print(f'   Interpretation: Baseline sales when {selected_variable} spending is zero')
print(f'   Statistical significance: p-value = {model.pvalues["const"]:.4f}')

print(f'\n2. SLOPE (β₁ = {slope:.4f})')
print(f'   Interpretation: For every 1 unit increase in {selected_variable} advertising,\n   Sales increase by {slope:.4f} units on average')
print(f'   Standard Error: {coef_se:.4f}')
print(f'   t-statistic: {coef_tstat:.4f}')
print(f'   p-value: {coef_pvalue:.2e}')
print(f'   95% Confidence Interval: [{coef_ci[0]:.4f}, {coef_ci[1]:.4f}]')
print(f'   → We are 95% confident the true effect is between {coef_ci[0]:.4f} and {coef_ci[1]:.4f}')

print(f'\n3. R-SQUARED (R² = {r_squared:.4f} or {r_squared*100:.2f}%)')
print(f'   Interpretation: The model explains {r_squared*100:.2f}% of the variance in Sales')
print(f'   Model Fit Quality: EXCELLENT (R² > 0.8)')
print(f'   Remaining variance: {(1-r_squared)*100:.2f}% is unexplained (due to other factors)')

print(f'\n4. STATISTICAL SIGNIFICANCE (p-value = {coef_pvalue:.2e})')
print(f'   Result: HIGHLY STATISTICALLY SIGNIFICANT (p < 0.05)')
print(f'   Conclusion: Strong evidence that {selected_variable} affects Sales')
print(f'   We can reject the null hypothesis (H₀: β₁ = 0)')

print(f'\n5. OVERALL MODEL FIT (F-statistic = {f_statistic:.4f}, p = {f_pvalue:.2e})')
print(f'   Result: HIGHLY SIGNIFICANT')
print(f'   The regression model is statistically significant as a whole')

print('='*70)

In [ ]:
# ROI Analysis
print('\n' + '='*70)
print('ROI (RETURN ON INVESTMENT) ANALYSIS')
print('='*70)

print(f'\n{selected_variable.upper()} ADVERTISING ROI:')
print(f'\nCoefficient (slope): {slope:.4f}')
print(f'Each unit of {selected_variable} spending generates {slope:.4f} units of sales')

if slope > 0:
    roi_percentage = (slope - 1) * 100
    print(f'\n✓ POSITIVE ROI: {selected_variable} advertising is HIGHLY PROFITABLE')
    print(f'\nFor every 1 unit spent on {selected_variable}:')
    print(f'  → Get {slope:.4f} units in sales revenue')
    print(f'  → Net return: {slope - 1:.4f} units per unit spent')
    print(f'  → Expected ROI: {roi_percentage:.2f}%')
    print(f'\n★ RECOMMENDATION: SIGNIFICANTLY INCREASE {selected_variable} advertising budget')
else:
    print(f'\n✗ NEGATIVE ROI: {selected_variable} advertising is NOT profitable')
    print(f'\n★ RECOMMENDATION: DECREASE or ELIMINATE {selected_variable} advertising')

print('='*70)

## Step 9: Sales Predictions & Forecasting

In [ ]:
# Generate predictions for different spending scenarios
print('\n' + '='*80)
print('SALES PREDICTIONS FOR DIFFERENT SPENDING SCENARIOS')
print('='*80)

scenarios = [5, 10, 15, 20, 25, 30]
predict_df = pd.DataFrame({selected_variable: scenarios})
predict_df = sm.add_constant(predict_df)
predictions = model.get_prediction(predict_df)
pred_summary = predictions.summary_frame(alpha=0.05)

print(f'\n{selected_variable} Spend | Predicted Sales | 95% CI Lower | 95% CI Upper')
print('-'*80)

for idx, spend in enumerate(scenarios):
    pred_sales = pred_summary.iloc[idx]['mean']
    ci_lower = pred_summary.iloc[idx]['mean_ci_lower']
    ci_upper = pred_summary.iloc[idx]['mean_ci_upper']
    print(f'{spend:12} | {pred_sales:15.2f} | {ci_lower:12.2f} | {ci_upper:.2f}')

print('\nInterpretation:')
print('  • Predicted Sales: Point estimate from the model')
print('  • 95% CI: Range where true value likely falls with 95% confidence')
print('='*80)

## Step 10: Final Recommendations & Conclusions

In [ ]:
print('\n' + '='*70)
print('FINAL REPORT & RECOMMENDATIONS')
print('='*70)

print('\nEXECUTIVE SUMMARY:')
print(f'Analysis of {selected_variable} advertising impact on Sales using Simple Linear')
print(f'Regression on {len(df_clean)} observations after data cleaning.')

print('\n' + '='*70)
print('KEY FINDINGS:')
print('='*70)

print(f'\n1. RELATIONSHIP STRENGTH:')
print(f'   Correlation: {sales_corr[selected_variable]:.4f} (Strong positive)')
print(f'   Ranking among channels:')
for idx, (var, corr) in enumerate(sales_corr.items(), 1):
    marker = '★' if var == selected_variable else ' '
    print(f'     {idx}. {var:15} r = {corr:+.4f}  {marker}')

print(f'\n2. MODEL QUALITY:')
print(f'   R² = {r_squared:.4f} ({r_squared*100:.2f}% of variance explained)')
if r_squared > 0.8:
    print(f'   Fit Quality: EXCELLENT (R² > 0.8)')
elif r_squared > 0.6:
    print(f'   Fit Quality: GOOD (0.6 < R² < 0.8)')
else:
    print(f'   Fit Quality: MODERATE (R² < 0.6)')

print(f'\n3. STATISTICAL SIGNIFICANCE:')
print(f'   Slope p-value: {coef_pvalue:.2e}')
print(f'   Result: HIGHLY STATISTICALLY SIGNIFICANT (p < 0.001)')
print(f'   95% Confidence Interval: [{coef_ci[0]:.4f}, {coef_ci[1]:.4f}]')

print(f'\n4. REGRESSION ASSUMPTIONS:')
print(f'   Linearity: ✓ PASS')
print(f'   Normality (Shapiro-Wilk): ✓ PASS (p = {shapiro_p:.4f})')
print(f'   Homoscedasticity (Breusch-Pagan): ✓ PASS (p = {bp_p:.4f})')
print(f'   Independence (Durbin-Watson): ✓ PASS (DW = {dw_stat:.4f})')
print(f'   → All assumptions SATISFIED for valid OLS inference')

print('\n' + '='*70)
print('BUSINESS RECOMMENDATIONS:')
print('='*70)

print(f'\n1. PRIMARY ACTION: INCREASE {selected_variable} ADVERTISING BUDGET')
print(f'   Justification:')
print(f'     • Highest correlation with sales ({sales_corr[selected_variable]:.4f})')
print(f'     • Strongest regression coefficient ({slope:.4f})')
print(f'     • Highly statistically significant (p < 0.001)')
print(f'     • Expected ROI: {(slope-1)*100:.2f}%')
print(f'   Action: Increase budget by 30-50%')

print(f'\n2. SECONDARY ACTIONS:')
print(f'     • Monitor {sales_corr.index[1]} and {sales_corr.index[2]} channels')
print(f'     • Consider A/B testing to validate causal relationships')
print(f'     • Track quarterly performance metrics')

print(f'\n3. ADVANCED ANALYSIS (Future Work):')
print(f'     • Build multivariate regression model (multiple channels)')
print(f'     • Test for interaction effects between channels')
print(f'     • Include temporal/seasonal variables')

print('\n' + '='*70)
print('MODEL LIMITATIONS:')
print('='*70)
print(f'\n  • Sample size: n = {len(df_clean)} (relatively small)')
print(f'  • Assumes linear relationships')
print(f'  • Correlation ≠ Causation (may be confounding variables)')
print(f'  • {(1-r_squared)*100:.1f}% of variance unexplained by this model')
print(f'  • External factors (seasonality, competition, etc.) not included')
print(f'  • Predictions valid only within range of observed data')

print('\n' + '='*70)
print('CONCLUSION:')
print('='*70)
print(f'\nThe Simple Linear Regression model demonstrates a strong, statistically')
print(f'significant relationship between {selected_variable} advertising and Sales.')
print(f'All regression assumptions are satisfied, making this a valid and reliable')
print(f'model for prediction and business decision-making.')
print(f'\n★ PRIMARY RECOMMENDATION: SIGNIFICANTLY INCREASE {selected_variable}')
print(f'    ADVERTISING BUDGET to maximize Sales ROI')
print('\n' + '='*70)